# Task 8.4 Qiskit Runtime Rest API

**Overview:** 이 노트북은 REST API를 통해 Qiskit Runtime에 접속하고, job을 제출하고, 결과를 불러오고, 세션을 관리하는 등의 사용법을 다룹니다.

In [1]:
# Setup: Import necessary libraries
import requests, json

print("Libraries imported successfully.")

Libraries imported successfully.


## Objective 1 : Interact with Qiskit IBM Runtime REST API

Qiskit Runtime REST API는 HTTP 프로토콜을 통해 IBM Quantum 시스템에 접근하는 인터페이스를 제공합니다.

* **회로 실행**: 양자컴퓨터에서 회로를 실행합니다.
* **Job과 session의 관리**: Job을 제출하고 불러오거나 취소, 삭제 등의 명령을 내릴 수 있습니다.
* **정보 불러오기**: 양자컴퓨터나 서비스 인스턴스, 실행 중인 워크로드, 계정 및 사용 정보 등을 불러올 수 있습니다.

### 접속

Qiskit Runtime REST API를 사용하기 위해서는 다음의 정보가 필요합니다:
1. **IBM Quantum Cloud API Key**: IBM Quantum Cloud에서 생성하는 API키
2. **IBM Quantum Instance CRN**: IBM Quantum service에서 사용할 수 있는 유효한 인스턴스의 CRN

접속 인증 과정은 다음과 같습니다:
1. API키를 제출하고 IAM 토큰을 받아옵니다.
2. IAM 토큰과 인스턴스 CRN을 사용해 API를 request합니다.

In [ ]:
url = 'https://iam.cloud.ibm.com/identity/token'
# Your IBM Cloud API key
api_key="<YOUR_API_KEY>"

data = f'grant_type=urn:ibm:params:oauth:grant-type:apikey&apikey={api_key}'
response = requests.post(url, data=data)

# Bearer token to authorize requests to the REST API
if response.status_code == 200:
        print(f"Access token obtained successfully.")
        bearer_token = response.json()['access_token']
else:
    print(f"Error obtaining bearer token:{response.status_code} ,{response.text}")
    bearer_token = None

Access token obtained successfully.


### 사용 가능한 백엔드 불러오기

이 예시에서는 여러분의 계정에서 사용 가능한 백엔드를 불러오는 방법을 보여줍니다.

In [ ]:
reqUrl = "https://quantum.cloud.ibm.com/api/v1/backends"
# Service CRN of your IBM Quantum instance
service_CRN = "<crn:YOUR_INSTANCE_CRN>"
 
headersList = {
  "Accept": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version":"2025-05-01"
}

payload = ""
 
response = requests.request("GET", reqUrl, data=payload,  headers=headersList)
 
if response.status_code == 200:
        print(f"Available Backends are:")
        print(response.json())
        # Selecting the first backend from the list
        backend = response.json()['devices'][0]['name']  

else:
    print(f"Error obtaining backends:{response.status_code} ,{response.text}")
    backend = None



Available Backends are:
{'devices': [{'name': 'ibm_boston', 'status': {'name': 'online', 'reason': 'available'}, 'qubits': 156, 'clops': {'type': 'hardware', 'value': 340000}, 'processor_type': {'family': 'Heron', 'revision': '3'}, 'queue_length': 0, 'performance_metrics': {'two_q_error_best': {'gate': 'cz', 'qubits': [43, 56], 'unit': '', 'value': 0.0005778036}, 'two_q_error_layered': {'unit': '', 'value': 0.0019291787}, 'two_q_error_median': {'unit': '', 'value': 0.0010932641}, 'readout_error_median': {'unit': '', 'value': 0.005126953}}}, {'name': 'ibm_fez', 'status': {'name': 'online', 'reason': 'available'}, 'qubits': 156, 'clops': {'type': 'hardware', 'value': 320000}, 'processor_type': {'family': 'Heron', 'revision': '2'}, 'queue_length': 0, 'performance_metrics': {'two_q_error_best': {'gate': 'cz', 'qubits': [2, 3], 'unit': '', 'value': 0.0014074539}, 'two_q_error_layered': {'unit': '', 'value': 0.0048554232}, 'two_q_error_median': {'unit': '', 'value': 0.0027981938}, 'readout_e

### Job 제출하기

이 예시에서는 sampler primitive를 이용해 job을 제출하고 양자 회로를 실행하는 방법을 보여줍니다.

<span style="color:red"> 아래의 셀을 실행하면 여러분의 계정으로 최대 **5초** 정도 양자 회로를 실행합니다.
</span>

In [4]:
reqUrl = "https://quantum.cloud.ibm.com/api/v1/jobs"

# Service CRN of your IBM Quantum instance
# bearer token obtained from IAM 
headersList = {
  "Accept": "application/json",
  "Content-Type": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version":"2025-05-01"
}

# Sample QASM circuit to be submitted 
circuit_qasm = '''OPENQASM 3.0; include "stdgates.inc"; bit[1] c; x $0; c[0] = measure $0;'''

payload = json.dumps({
  "program_id": "sampler",
  # Target backend
  "backend": f"{backend}",
  "params": {
    "pubs": [[
      circuit_qasm
    ]],
    "version": 2
  }
})
 
response = requests.request("POST", reqUrl, data=payload,  headers=headersList)

if response.status_code == 200:
        print(f"Submitted Job:")
        print(response.json())
        job_id = response.json()['id']
else:
    print(f"Error submitting job:{response.status_code} ,{response.text}")
    job_id = None



Submitted Job:
{'id': 'd7ol82edvl0c738nv9n0', 'backend': 'ibm_boston'}


### 결과 불러오기

이 예시에서는 job을 제출한 후 job의 상태를 확인하고 결과를 불러오는 방법을 보여줍니다.

In [ ]:
# check job_id obtained from job submission
reqUrl = f"https://quantum.cloud.ibm.com/api/v1/jobs/{job_id}"
results_reqUrl = f"https://quantum.cloud.ibm.com/api/v1/jobs/{job_id}/results"
 
headersList = {
  "Accept": "application/json",
  "Content-Type": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version":"2025-05-01"
}

payload = ""

response = requests.request("GET", reqUrl, data=payload,  headers=headersList)

if response.status_code == 200:
        print(f"Job Status: {response.json()['status']}")
        # Check if job is completed to retrieve results
        if response.json()['status'] == 'Completed':
            print("Retrieving Results:")
            results_response = requests.request("GET", results_reqUrl, data=payload,  headers=headersList)
            if results_response.status_code == 200:
                print(f"Job Results:")
                print(results_response.json())
            else:
                print(f"Error retrieving job results:{results_response.status_code} ,{results_response.text}")
        elif response.json()['status'] == 'Failed':
            print("Job has failed. No results to retrieve.")
        else:
            print(f"Job is {response.json()['status']}")
else:
    print(f"Error retrieving job status:{response.status_code} ,{response.text}")

Job Status: Completed
Retrieving Results:
Job Results:
{'results': [{'data': {'c': {'samples': ['0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '

### Session 생성하기

이 예시에서는 session을 생성하고 session ID를 받아오는 방법을 보여줍니다.

<span style="color:red"> **참고:** 오픈 플랜 유저는 세션 모드를 사용할 수 없습니다. 오픈 플랜으로 접속하는 경우, 아래의 코드는 실패하게 됩니다. </span>

In [11]:
reqUrl = "https://quantum.cloud.ibm.com/api/v1/sessions"
 
headersList = {
  "Accept": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "Content-Type": "application/json"
}
 
payload = json.dumps({
  "mode": "dedicated",
  "max_ttl": 28800
})
 
response = requests.request("POST", reqUrl, data=payload,  headers=headersList)

if response.status_code == 200:
  sessionId = response.json()['id']
  print(response.json())
  print(sessionId)
else:
    print(f"Error creating session:{response.status_code} ,{response.text}")


{'id': 'c045383a-21e6-472e-b106-381830efd999', 'backend_name': '', 'interactive_ttl': 60, 'max_ttl': 28800, 'active_ttl': 28800, 'state': 'open', 'accepting_jobs': True, 'mode': 'dedicated', 'user_id': 'IBMid-6960006XIF'}
c045383a-21e6-472e-b106-381830efd999


### Session에 Job 제출하기

위에서 가져온 session ID를 사용하여 아래와 같이 job을 실행할 수 있습니다.

<span style="color:red"> 세션이 정상적으로 작동하는 경우, 아래의 셀을 실행하면 여러분의 계정으로 **5초** 정도 양자 회로를 실행합니다. 하지만 세션은 job이 완료된 후에도 약 1분간 양자컴퓨터를 점유하므로 곧바로 다음의 session 닫기 코드를 실행해주세요.
</span>

In [12]:
reqUrl = "https://quantum.cloud.ibm.com/api/v1/jobs"
 
headersList = {
  "Accept": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version": "2025-05-01",
  "Content-Type": "application/json"
}

payload = json.dumps({
  "program_id": "sampler",
  "backend": f"{backend}",
  "session_id": f"{sessionId}",
  "params": {
    "pubs": [[
      "OPENQASM 3.0; include \"stdgates.inc\"; bit[1] c; x $0; c[0] = measure $0;"
    ]],
    "options": {},
    "version": 2
  }
})
 
response = requests.request("POST", reqUrl, data=payload,  headers=headersList)
if response.status_code == 200:
  print(f"Submitted Job:")
  print(response.json())
  session_job_id = response.json()['id']
else:
  print(f"Error submitting job:{response.status_code} ,{response.text}")

Submitted Job:
{'id': 'd7olet8ror3c73c04070', 'backend': 'ibm_boston', 'session_id': 'c045383a-21e6-472e-b106-381830efd999'}


In [ ]:
# check job_id obtained from job submission
reqUrl = f"https://quantum.cloud.ibm.com/api/v1/jobs/{session_job_id}"
results_reqUrl = f"https://quantum.cloud.ibm.com/api/v1/jobs/{session_job_id}/results"
 
headersList = {
  "Accept": "application/json",
  "Content-Type": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version":"2025-05-01"
}

payload = ""

response = requests.request("GET", reqUrl, data=payload,  headers=headersList)

if response.status_code == 200:
        print(f"Job Status: {response.json()['status']}")
        # Check if job is completed to retrieve results
        if response.json()['status'] == 'Completed':
            print("Retrieving Results:")
            results_response = requests.request("GET", results_reqUrl, data=payload,  headers=headersList)
            if results_response.status_code == 200:
                print(f"Job Results:")
                print(results_response.json())
            else:
                print(f"Error retrieving job results:{results_response.status_code} ,{results_response.text}")
        elif response.json()['status'] == 'Failed':
            print("Job has failed. No results to retrieve.")
        else:
            print(f"Job is {response.json()['status']}")
else:
    print(f"Error retrieving job status:{response.status_code} ,{response.text}")

Job Status: Completed
Retrieving Results:
Job Results:
{'results': [{'data': {'c': {'samples': ['0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '0x1', '

### Session 닫기

REST API를 통해 session을 닫는 방법은 다음과 같습니다.

In [19]:
reqUrl = f"https://quantum.cloud.ibm.com/api/v1/sessions/{sessionId}/close"

headersList = {
  "Accept": "application/json",
  "Authorization": f"Bearer {bearer_token}",
  "Service-CRN": f"{service_CRN}",
  "IBM-API-Version": "2025-05-01",
}

response = requests.request("DELETE", reqUrl, headers=headersList)
if response.status_code == 204:
  print(f"Closed Session {sessionId}")
else:
  print(f"Error submitting job:{response.status_code} ,{response.text}")

Closed Session c045383a-21e6-472e-b106-381830efd999


---
## 요약
---

이 노트북에서는 다음의 내용을 다루었습니다:

## Qiskit Runtime REST API:

1. **Qiskit Runtime REST API**: Qiskit Runtime REST API를 통해 회로를 실행하고, job을 제출하거나 불러오고, 백엔드, 워크로드 등의 정보를 불러올 수 있습니다.
2. **접속 인증이 필요**: API키를 제출하여 bearer 토큰을 받아오고, bearer 토큰과 서비스 인스턴스 CRN을 사용하여 API에 요청을 제출할 수 있습니다.
3. **Job 제출**: QASM3 회로와 primitive, 백엔드를 지정하여 job을 제출할 수 있습니다.
4. **Job 상태 확인**: Job 상태를 확인하고 job이 완료되거나 실패하면 결과를 불러올 수 있습니다.
5. **Session**: Session을 생성하고 session ID를 이용해 job을 실행할 수 있습니다.

---

## 연습 문제

**1) Why might the code below fail?**

```
import requests
import json
 
reqUrl = "https://quantum.cloud.ibm.com/api/v1/jobs"

headersList = {
  "Accept": "application/json",
  "Content-Type": "application/json",
  "Authorization": f"Bearer {My_API_token}",
  "Service-CRN": f"{My_Service_CRN}",
  "IBM-API-Version":"2025-05-01"
}

circuit_qasm = '''OPENQASM 3.0; include "stdgates.inc"; bit[1] c; x $0; c[0] = measure $0;'''
 
payload = json.dumps({
  "program_id": "sampler",
  "backend": f"{backend}",
  "params": {
    "pubs": [[
      circuit_qasm
    ]],
    "version": 2
  }
})
 
response = requests.request("POST", reqUrl, data=payload,  headers=headersList)
print(response.json())
job_id = response.json()['id']
```


A) request URL should have the job id

B) The token must be passed as a query parameter

C) The authorization header should have the bearer token not the API token

D) program_id is not required


***정답:***
<Details>
C) REST API에서는 bearer(IAM) 토큰을 사용하기 때문에 API를 호출할 때 IBM Quantum API 토큰이 아닌 bearer 토큰을 사용해야 합니다.
<br/>
</Details>

---

**2) What information will be returned from the request below?**

```
curl -X GET https://quantum.cloud.ibm.com/api/v1/jobs/12345 \
  -H "Authorization: Bearer BEARER_TOKEN",
  -H "Service-CRN": f"{service_CRN}"
```

A) Job results only

B) Backend calibration data

C) The transpiled circuit and its execution parameters

D) Job metadata and execution status

***정답:***
<Details>
D) Job을 불러오면 job의 상태와 백엔드 정보, 생성 시간, 실행 시간 등의 메타데이터와 실행 상태 등의 정보를 모두 불러올 수 있습니다.
<br/>
</Details>

---